# Assignment 07 – Interactive Visualizations

**Contributors:** 

Danny Schönhals

Athithya Mariyanayagam

---
## Task 1 – Autonomous Vehicle Survey of Bicyclists and Pedestrians in Pittsburgh

In [1]:
import pandas as pd
import plotly.graph_objects as go

In [2]:
df = pd.read_csv('data/av_survey_data/avsurvey2019data.csv', encoding='latin-1')
df2 = df[['SafeAv', 'ArizonaCrash', 'FamiliarityNews']].dropna()
df = df[['SafeAv', 'Age', 'BikePghMember']].dropna() # Drop rows with missing values in the specified columns
df['SafeAv'] = df['SafeAv'].astype(int) # Convert 'SafeAv' to integer type
df2['SafeAv'] = df2['SafeAv'].astype(int) # Convert 'SafeAv' to integer type
df.head()
print(df2['ArizonaCrash'].unique())
print(df2['FamiliarityNews'].unique())

<ArrowStringArray>
[                          'No change', 'Significantly more negative opinion',
      'Somewhat more negative opinion', 'Significantly more positive opinion',
      'Somewhat more positive opinion']
Length: 5, dtype: str
<ArrowStringArray>
['To a moderate extent',    'To a large extent',     'To little extent',
           'Not at all',       'To some extent']
Length: 5, dtype: str


In [3]:
age_order        = ['Under 18', '18-24', '25-34', '35-44', '45-54', '55-64', '65+'] # Define the desired order of age groups
membership_opts  = ['All', 'Yes', 'No', 'Not sure'] # Membership status options
safeav_values    = [1, 2, 3, 4, 5] # Possible values for 'SafeAv' (1 to 5)
colors           = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A'] # Colors for each 'SafeAv' value

fig = go.Figure()

for member in membership_opts: # Loop through each membership status option
    df_f   = df if member == 'All' else df[df['BikePghMember'] == member]
    totals = df_f.groupby('Age').size().reindex(age_order, fill_value=0)

    for i, sav in enumerate(safeav_values): # Loop through each 'SafeAv' value
        counts = df_f[df_f['SafeAv'] == sav].groupby('Age').size().reindex(age_order, fill_value=0)
        pcts   = (counts / totals * 100).round(1).fillna(0)

        fig.add_trace(go.Bar(
            x=age_order,
            y=counts.values,
            name=f'SafeAv {sav}',
            marker_color=colors[i],
            legendgroup=str(sav),
            showlegend=(member == 'All'),
            customdata=pcts.values,
            hovertemplate='Age: %{x}<br>Count: %{y}<br>Share: %{customdata}%<extra></extra>',
            visible=(member == 'All')
        ))

n = len(safeav_values) # Number of 'SafeAv' categories, used to calculate visibility for each membership option
buttons = [] # Create buttons for each membership status to toggle visibility of corresponding bars
for i, member in enumerate(membership_opts): # Loop through each membership status option to create a button
    vis = [False] * (len(membership_opts) * n)
    for j in range(n):
        vis[i * n + j] = True
    buttons.append(dict(
        label=member,
        method='update',
        args=[{'visible': vis}, {'title': f'SafeAv by Age Group – BikePgh: {member}'}]
    ))

fig.update_layout(
    updatemenus=[dict(
        buttons=buttons,
        direction='down',
        x=0.0, xanchor='left',
        y=1.15, yanchor='top'
    )],
    barmode='stack',
    title='SafeAv by Age Group – BikePgh: All',
    xaxis_title='Age Group',
    yaxis_title='Count',
    legend_title='SafeAv'
)
fig.show()

The 25-44 age groups have the most respondents, while 18-24 has only 32 participants. Ratings 3 to 5 dominate across all age groups. Since the y-axis shows absolute counts, bar height differences primarily reflect sample size. The hover percentages reveal a slight trend: younger respondents (18-24: 79% rate 4-5) tend to view AV safety more positively than older ones (55-64: 54% rate 4-5), but this can only be read from the percentages, not the bar heights alone.

In [4]:
# New Visualization: SafeAv by Familiarity with AV News
import plotly.express as px

df2_plot = df2.copy()
df2_plot['ArizonaCrash'] = pd.Categorical(
    df2_plot['ArizonaCrash'],
    categories=['Significantly more positive opinion', 'Somewhat more positive opinion',
                'No change',
                'Somewhat more negative opinion', 'Significantly more negative opinion'],
    ordered=True
)
df2_plot['FamiliarityNews'] = pd.Categorical(
    df2_plot['FamiliarityNews'],
    categories=['To a large extent', 'To some extent', 'To a moderate extent', 'To little extent', 'Not at all'],
    ordered=True
)

fig = px.parallel_categories(
    df2_plot,
    dimensions=['FamiliarityNews', 'ArizonaCrash', 'SafeAv'],
    color='SafeAv',
    color_continuous_scale=px.colors.sequential.Viridis,
    labels={
        'FamiliarityNews': 'Familiarity: AV News',
        'ArizonaCrash': 'Arizona Crash Impact',
        'SafeAv': 'Safety Rating'
    },
    title='SafeAv by Familiarity with AV News – Parallel Categories'
)
fig.update_layout(coloraxis_colorbar=dict(title='SafeAv'))
fig.show()

---
## Task 2 – Degradation Measurement of Robot Arm Position Accuracy

In [13]:
import pandas as pd
import plotly.graph_objects as go

In [14]:
df_robot = pd.read_csv('data/robot_position_accuracy/ur5_combined.csv')
print(f'Shape: {df_robot.shape}')

Shape: (153658, 77)


In [ ]:
group_cols = ['cold_start', 'speed', 'payload_lb', 'trial_number']

records = []
for keys, grp in df_robot.groupby(group_cols):
    row = dict(zip(group_cols, keys))
    row['max_dev_position'] = (grp['robot_target_joint_positions_j1'] - grp['robot_actual_joint_positions_j1']).abs().max() # calculating max deviation for joint 1 as an example
    row['max_dev_velocity'] = (grp['robot_target_joint_velocities_j1'] - grp['robot_actual_joint_velocities_j1']).abs().max() # calculating maxdeviation for velocity for joint 1 as an example
    row['max_dev_current']  = (grp['robot_target_joint_current_j1']    - grp['robot_actual_joint_current_j1']).abs().max() # calculating max deviation for current for joint 1 as an example
    records.append(row)

df_dev = pd.DataFrame(records)
df_dev

,cold_start,speed,payload_lb,trial_number,max_dev_position,max_dev_velocity,max_dev_current
0,False,fullspeed,1.6,1,0.069401,0.031808,1.147812
1,False,fullspeed,1.6,2,0.083794,0.031675,1.030445
2,False,fullspeed,1.6,3,0.086501,0.030111,1.123090
3,False,fullspeed,4.5,1,0.089630,0.029050,1.275706
4,False,fullspeed,4.5,2,0.064774,0.038100,1.112017
5,False,fullspeed,4.5,3,0.105228,0.029763,1.268193
6,False,halfspeed,1.6,1,0.050225,0.014085,0.781889
7,False,halfspeed,1.6,2,0.048450,0.016616,0.681211
8,False,halfspeed,1.6,3,0.055948,0.010573,0.729925
9,False,halfspeed,4.5,1,0.058823,0.018981,0.964372


In [16]:
color_map = {'fullspeed': '#EF553B', 'halfspeed': '#636EFA'}

fig = go.Figure()
for speed, grp in df_dev.groupby('speed'):
    fig.add_trace(go.Scatter3d(
        x=grp['max_dev_position'],
        y=grp['max_dev_velocity'],
        z=grp['max_dev_current'],
        mode='markers',
        name=speed,
        marker=dict(size=8, color=color_map[speed]),
        hovertemplate='Position: %{x:.5f}<br>Velocity: %{y:.5f}<br>Current: %{z:.5f}<extra></extra>'
    ))

fig.update_layout(
    title='Max Deviation Joint 1 – Position vs Velocity vs Current',
    scene=dict(
        xaxis_title='Max dev position',
        yaxis_title='Max dev velocity',
        zaxis_title='Max dev current'
    ),
    legend_title='Speed'
)
fig.show()

**Chart choice and interpretation:**

A 3D scatter plot was chosen because it shows the relationship between all three variables simultaneously in one interactive view that can be rotated freely. Each point represents one trial, colored by speed condition. The plot shows that fullspeed trials (red) tend to have higher velocity and current deviations than halfspeed trials (blue), suggesting that speed is the main driver of deviation. Position deviation remains small and shows no clear separation between the two groups.